<a href="https://colab.research.google.com/github/lucas6028/x-coach/blob/main/notebooks/run_rehab24_videomae_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REHAB24-6 × VideoMAE 特徵提取 (Colab)

在 Colab GPU 上對 **REHAB24-6** 每個動作重複 (repetition) 提取 VideoMAE 特徵。

流程：
1. 確認 GPU
2. 掛載 Google Drive（**原始碼已上傳到 Drive**）
3. 直接從 [Zenodo](https://zenodo.org/records/13305826) 下載 `videos.zip` + `Segmentation.csv`
4. 解壓縮並偵測資料夾結構
5. 建立 manifest / splits / labels (`scripts/rehab24/build_manifest.py`)
6. 跑 VideoMAE 特徵提取 (`scripts/rehab24/extract_videomae_features.py`)
7. 打包特徵存回 Drive

> VideoMAE 只需要影片 (`videos.zip`) 與切割標註 (`Segmentation.csv`)，因此 manifest 用 `--skip-path-validation` 略過尚未下載的 skeleton `.npy`。
> 之後若要跑 skeleton / fusion，再回 Zenodo 下載 `2d_joints.zip` / `3d_joints.zip` 即可。

### 0. 確認 GPU（Runtime → Change runtime type → GPU）

In [ ]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

### 1. 掛載 Google Drive（原始碼）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. 設定路徑

把 `REPO_ROOT` 改成你在 Drive 上放原始碼的位置（裡面要有 `scripts/` 和 `src/`）。

In [ ]:
import os

# === 原始碼位置（已上傳到 Google Drive，請改成你的路徑）===
REPO_ROOT = "/content/drive/MyDrive/x-coach"

# === 下載與解壓縮位置（Colab 本機磁碟，讀寫快）===
DOWNLOAD_DIR = "/content/drive/MyDrive/x-coach/data/REHAB24-6"
DATA_ROOT = DOWNLOAD_DIR  # 解壓後第 4 步會自動偵測並更新
SEG_PATH = os.path.join(DOWNLOAD_DIR, "Segmentation.csv")

# === manifest / splits / labels 存回 Drive，可長期保存 ===
PROCESSED_ROOT = "/content/drive/MyDrive/x-coach/data/REHAB24-6/processed"

# === VideoMAE 特徵先寫到本機，最後再打包回 Drive ===
LOCAL_FEATURE_DIR = "/content/videomae_features"
DRIVE_FEATURE_ZIP = "/content/drive/MyDrive/x-coach/data/REHAB24-6/videomae_features.zip"

# 兩個相機視角；只要單一視角可改成 "cam17"
CAMERAS = "cam17,cam18"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(PROCESSED_ROOT, exist_ok=True)
os.makedirs(LOCAL_FEATURE_DIR, exist_ok=True)
assert os.path.isdir(os.path.join(REPO_ROOT, "scripts", "rehab24")), \
    f"在 {REPO_ROOT} 找不到 scripts/rehab24，請確認 REPO_ROOT 設定正確"
print("REPO_ROOT     :", REPO_ROOT)
print("DOWNLOAD_DIR  :", DOWNLOAD_DIR)
print("PROCESSED_ROOT:", PROCESSED_ROOT)

### 3. 從 Zenodo 下載資料集

`videos.zip` 約 **2.7 GB**，`Segmentation.csv` 約 53 KB。`wget -c` 可續傳，重跑不會重新下載。

In [ ]:
!wget -c -O "{DOWNLOAD_DIR}/videos.zip" "https://zenodo.org/records/13305826/files/videos.zip?download=1"
!wget -c -O "{SEG_PATH}" "https://zenodo.org/records/13305826/files/Segmentation.csv?download=1"
!ls -lh "{DOWNLOAD_DIR}"

### 4. 解壓縮並偵測資料夾結構

manifest 內的影片路徑形如 `ExN/<video_id>-Camera17-30fps.mp4`，所以 `--data-root` 必須是 `ExN` 資料夾的**上一層**；下面會自動偵測。

In [ ]:
!unzip -q -o "{DOWNLOAD_DIR}/videos.zip" -d "{DOWNLOAD_DIR}"

In [ ]:
from pathlib import Path

cam17 = list(Path(DOWNLOAD_DIR).rglob("*-Camera17-30fps.mp4"))
cam18 = list(Path(DOWNLOAD_DIR).rglob("*-Camera18-30fps-transposed.mp4"))
print(f"找到 {len(cam17)} 支 Camera17、{len(cam18)} 支 Camera18 影片")
assert cam17, "找不到影片，請確認 videos.zip 已正確解壓縮"

# data-root = ExN 資料夾的上一層
DATA_ROOT = str(cam17[0].parent.parent)
print("偵測到的 DATA_ROOT:", DATA_ROOT)
print("範例影片:", cam17[0])

### 5. 建立 manifest / splits / labels

依 person id 做固定切割：train `{1,2,3,4,5,7,10}` · val `{6}` · test `{8,9}`。每個重複 × 相機 = 一個 sample。

In [ ]:
!python "{REPO_ROOT}/scripts/rehab24/build_manifest.py" \
  --data-root "{DATA_ROOT}" \
  --segmentation "{SEG_PATH}" \
  --processed-root "{PROCESSED_ROOT}" \
  --cameras "{CAMERAS}" \
  --skip-path-validation

驗證 manifest 引用的影片都存在（跑特徵提取前先確認，避免長時間跑到一半才報錯）。

In [ ]:
import sys
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from pathlib import Path
from src.rehab24.dataset import load_manifest, resolve_data_path

rows = load_manifest(Path(PROCESSED_ROOT) / "manifest.csv")
missing = [r["sample_id"] for r in rows
           if not resolve_data_path(Path(DATA_ROOT), r["video_path"]).exists()]
print(f"manifest 共 {len(rows)} 列，缺少影片 {len(missing)} 個")
for s in missing[:10]:
    print("  missing:", s)
assert not missing, "有影片找不到——若只想用單一視角，請把 CAMERAS 改成 'cam17' 後重跑第 5 步"

### 6. 安裝套件

Colab 已內建 `torch` 與 `opencv`，只需補裝 transformers 相關套件。

In [ ]:
!pip install -q transformers accelerate timm

### 7. 煙霧測試 (smoke test)

先用 `--limit 4` 確認模型載入、影片讀取、寫檔都正常，再跑完整資料集。

In [ ]:
!python "{REPO_ROOT}/scripts/rehab24/extract_videomae_features.py" \
  --data-root "{DATA_ROOT}" \
  --manifest "{PROCESSED_ROOT}/manifest.csv" \
  --output-dir "{LOCAL_FEATURE_DIR}" \
  --num-clips 4 \
  --device cuda \
  --limit 4

### 8. 完整特徵提取

輸出寫到本機 `LOCAL_FEATURE_DIR/{split}/{sample_id}.npz`。已存在的檔會跳過（加 `--overwrite` 可強制重算），所以 Colab 斷線後可直接重跑接續。

In [ ]:
!python "{REPO_ROOT}/scripts/rehab24/extract_videomae_features.py" \
  --data-root "{DATA_ROOT}" \
  --manifest "{PROCESSED_ROOT}/manifest.csv" \
  --output-dir "{LOCAL_FEATURE_DIR}" \
  --num-clips 4 \
  --device cuda

### 9. 打包並存回 Google Drive

幾千個小 `.npz` 直接寫 Drive 很慢，所以前面寫本機、這裡再壓成一個 zip 存回 Drive。

In [ ]:
import os
from pathlib import Path
n = len(list(Path(LOCAL_FEATURE_DIR).rglob("*.npz")))
print(f"共產生 {n} 個特徵檔")
os.makedirs(os.path.dirname(DRIVE_FEATURE_ZIP), exist_ok=True)

In [ ]:
!cd "{LOCAL_FEATURE_DIR}" && zip -r -q "{DRIVE_FEATURE_ZIP}" .
!ls -lh "{DRIVE_FEATURE_ZIP}"

### 下一步

- 解壓 `videomae_features.zip` 後，可接 `scripts/rehab24/fuse_features.py`（需先有 skeleton 特徵）或直接餵 `scripts/rehab24/train_correctness_classifier.py`：
  ```bash
  python scripts/rehab24/train_correctness_classifier.py \
    --feature-dir videomae_features --manifest manifest.csv \
    --train-keys splits/train_keys.json --val-keys splits/val_keys.json \
    --test-keys splits/test_keys.json --labels labels/correctness.json --device cuda
  ```
- `manifest.csv`、`splits/`、`labels/` 已存在 `PROCESSED_ROOT`（Drive），不需重建。